# 03. 리뷰 텍스트 전처리 (T1, 2026-10-06)

**목적**: 리뷰 본문을 감정·원인 분석(Attri et al., 2025, *Findings of EMNLP*)에 넣을 수 있는 형태로 정리한다. 기준은 **논문 방식을 최대한 따른다.**

논문에서 가져온 전처리 기준 (쪽수는 Findings of EMNLP 2025 기준)
| 논문 | 이 노트북 |
|---|---|
| "Preprocessing steps include **standardizing** product titles and **review texts**, exact span matching, and **deduplication**" (p.5521) | HTML 태그·특수문자 코드·URL·공백을 정리한 `text_clean`을 만든다. 중복은 9/30에 이미 제거했고, 여기서는 남은 중복을 **세어서 표시**만 한다 |
| 리뷰 길이 $10 \le len(r) \le 100$ 만 사용 (장황함 통제, p.5519) | 단어 수 `n_words`를 세고 `in_len_range` 열로 **표시**한다. 실제로 거르는 건 다음 노트북(표본 추출)에서 한다 |
| 평점·도움돼요 수는 편향 때문에 모델 입력에서 제외 (p.5519) | 모델에 넣는 건 `text_clean`뿐. 평점 등은 분석용으로 열만 남긴다 |
| 출력에 `review_id`가 있음 (부록 D.8) | 리뷰마다 고정 ID `review_id`를 만든다 |
| trigger는 리뷰 원문의 **정확한 부분 문자열** (p.5517) | 그래서 **대소문자·문장부호·불용어는 건드리지 않는다.** 바꾸면 trigger를 원문에서 찾을 수 없다 |

**입력**: `data/interim/filtered_reviews/reviews_filtered.parquet` (02 노트북 결과, 1,107,639건)
**출력**: `data/interim/text/reviews_text.parquet` (+ 확인용 예시·요약 표)

**실행**: VS Code에서 열고 커널 `.venv` 확인 → **Run All** → 끝나면 **Ctrl+S**. 출력 중 **[점검]** 표시가 있는 표를 Claude에게 보내서 확인받는다.

## 1. 준비

- 처음 쓰는 패키지는 없다 (duckdb, pandas, matplotlib).
- `SET TimeZone = 'UTC'`: 02와 같은 이유 (리뷰 시각은 UTC).

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
pd.set_option("display.max_colwidth", 200)

DATA = Path("../data")
IN_PATH  = DATA / "interim/filtered_reviews/reviews_filtered.parquet"
OUT_DIR  = DATA / "interim/text"
OUT_PATH = OUT_DIR / "reviews_text.parquet"
OUT_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.sql("SET TimeZone = 'UTC'")
con.sql("SET preserve_insertion_order = false")
con.sql(f"CREATE OR REPLACE VIEW raw AS SELECT * FROM read_parquet('{IN_PATH.as_posix()}')")
print("입력 리뷰 수:", con.sql("SELECT count(*) FROM raw").fetchone()[0])

## 2. 원문에 어떤 '지저분한 것'이 얼마나 있나 (정리 전 점검)

정리 규칙을 정하기 전에, 본문에 실제로 무엇이 섞여 있는지 센다.
- `<br />` 등 **HTML 태그**: Amazon 리뷰는 줄바꿈이 `<br />`로 저장돼 있다.
- `&amp;` `&quot;` `&#34;` 같은 **HTML 특수문자 코드**: `&`, `"` 등이 코드로 저장된 것.
- **URL**: 링크 주소. 감정과 무관하고 단어 수를 부풀린다.
- **빈 리뷰**: 본문이 없거나 공백만 있음.

In [ ]:
con.sql("""
    SELECT
        count(*) AS 전체,
        sum(CASE WHEN text IS NULL OR trim(text) = '' THEN 1 ELSE 0 END) AS 빈_본문,
        sum(CASE WHEN regexp_matches(text, '<[^>]+>') THEN 1 ELSE 0 END) AS HTML태그_포함,
        sum(CASE WHEN regexp_matches(text, '&(amp|quot|lt|gt|nbsp|#[0-9]+);') THEN 1 ELSE 0 END) AS 특수문자코드_포함,
        sum(CASE WHEN regexp_matches(text, '(https?://|www\\.)') THEN 1 ELSE 0 END) AS URL_포함
    FROM raw
""").df()

## 3. 본문 정리 (`text_clean`)

논문의 "review text standardizing"에 해당한다. **의미는 바꾸지 않고 형식만** 정리한다. 순서가 중요하다.
1. `<br>`, `<br />`, `<p>` 같은 줄바꿈 태그 → **공백** (붙어 있던 문장이 이어 붙지 않게)
2. 나머지 HTML 태그(`<a ...>`, `<b>` 등) 삭제
3. URL 삭제 (`http://…`, `https://…`, `www.…`)
4. 특수문자 코드를 원래 글자로: `&amp;`→`&`, `&quot;` `&#34;`→`"`, `&#39;`→`'`, `&lt;`→`<`, `&gt;`→`>`, `&nbsp;`→공백
5. 여러 칸 공백·줄바꿈·탭을 공백 하나로, 앞뒤 공백 제거

**하지 않는 것**: 소문자 변환, 문장부호·숫자·이모지 제거, 불용어 제거, 표제어 변환, 오타 수정. 이유는 두 가지다. ① LLM은 원래 문장 그대로 읽을 때 감정을 더 잘 파악한다. ② trigger는 `text_clean`의 정확한 부분 문자열이어야 하는데, 글자를 바꾸면 원문과 대조할 수 없다. (TF-IDF 같은 단어 빈도 분석용 정리는 EDA 노트북에서 따로 한다.)

리뷰 **제목**(`title`)도 같은 방식으로 `title_clean`을 만든다. 논문 예시는 본문만 모델에 넣으므로 제목은 EDA용으로만 남긴다.

In [ ]:
CLEAN = """
trim(regexp_replace(
  replace(replace(replace(replace(replace(replace(replace(
    regexp_replace(
      regexp_replace(
        regexp_replace(coalesce({col}, ''), '<\\s*(br|p|/p|div|/div)\\s*/?\\s*>', ' ', 'gi'),
      '<[^>]+>', '', 'g'),
    '(https?://|www\\.)\\S+', ' ', 'gi'),
  '&amp;', '&'), '&quot;', '"'), '&#34;', '"'), '&#39;', ''''), '&lt;', '<'), '&gt;', '>'), '&nbsp;', ' '),
'\\s+', ' ', 'g'))
"""
text_clean  = CLEAN.format(col="text")
title_clean = CLEAN.format(col="title")
print(text_clean)

## 4. 리뷰 ID, 주, 단어 수, 표시 열 만들기

- `review_id`: `user_id + asin + timestamp`를 해시(md5)한 앞 16자리. 이 세 개가 같으면 같은 리뷰라서(9/30 중복 제거 기준과 같은 축) 고정 ID로 쓸 수 있다. 감정 분석 결과를 원래 리뷰에 다시 붙일 때 쓴다.
- `week`: 그 주 일요일 날짜 (팀 기준, 02 노트북과 같은 계산).
- `n_words`: 공백으로 나눈 단어 수. 논문의 리뷰 길이 len(r)에 해당한다.
- `in_len_range`: 10 ≤ 단어 수 ≤ 100 이면 True (논문 표본 조건). **여기서 거르지 않고 표시만** 한다. 길이 분포를 EDA에서 보여 주고, 걸러서 몇 %가 남는지도 보고하기 위해서다.
- `is_empty`: 정리 후 본문이 비었으면 True. 리뷰 수·평점 집계에는 쓰지만 감정 분석에서는 뺀다 (명세 5장).
- `non_ascii_share`: 영어 알파벳·숫자·기본 기호가 아닌 글자(이모지, 악센트 문자 등)의 비율. 0.3 이상이면 `high_non_ascii`로 표시한다. 이모지만 가득한 리뷰처럼 **LLM이 읽기 어려운 글**을 찾는 용도다.
  - 주의: 이건 언어 판별이 아니다. 스페인어처럼 알파벳을 쓰는 언어는 비율이 낮아 안 잡힌다. 110만 건 전체에 언어 판별기를 돌리면 오래 걸리므로, **정확한 영어 판별은 다음 노트북에서 뽑은 표본(수백 건)에만** 한다.

In [ ]:
con.sql(f"""
    CREATE OR REPLACE TABLE txt AS
    WITH c AS (
        SELECT *,
               {text_clean}  AS text_clean,
               {title_clean} AS title_clean
        FROM raw
    )
    SELECT
        substr(md5(user_id || '|' || asin || '|' || CAST(timestamp AS VARCHAR)), 1, 16) AS review_id,
        asin, parent_asin, product_type, user_id,
        review_date,
        CAST(date_trunc('week', review_date + INTERVAL 1 DAY) - INTERVAL 1 DAY AS DATE) AS week,
        rating, helpful_vote, verified_purchase,
        title_clean, text_clean,
        CASE WHEN text_clean = '' THEN 0 ELSE len(string_split(text_clean, ' ')) END AS n_words,
        text_clean = '' AS is_empty,
        CASE WHEN length(text_clean) = 0 THEN 0
             ELSE 1 - length(regexp_replace(text_clean, '[^\\x20-\\x7E]', '', 'g')) / length(text_clean) END AS non_ascii_share
    FROM c
""")
con.sql("""
    ALTER TABLE txt ADD COLUMN in_len_range BOOLEAN;
    UPDATE txt SET in_len_range = n_words BETWEEN 10 AND 100;
    ALTER TABLE txt ADD COLUMN high_non_ascii BOOLEAN;
    UPDATE txt SET high_non_ascii = non_ascii_share >= 0.3;
""")
print("행 수:", con.sql("SELECT count(*) FROM txt").fetchone()[0])
dup_id = con.sql("SELECT count(*) - count(DISTINCT review_id) FROM txt").fetchone()[0]
print("review_id 중복:", dup_id, "(0이어야 함)")

## 5. [점검] 정리 전후 비교 예시

규칙이 의도대로 동작하는지 눈으로 확인한다. HTML 태그·특수문자 코드·URL이 있던 리뷰를 종류별로 3건씩 뽑아 원문과 정리본을 나란히 보여 준다.
**이 표를 Claude에게 보내서 확인받는다.** 의미가 바뀐 곳이 있으면 규칙을 고친다.

In [ ]:
examples = con.sql(f"""
    WITH j AS (
        SELECT DISTINCT r.text AS 원문, t.text_clean AS 정리본,
               CASE WHEN regexp_matches(r.text, '<[^>]+>') THEN 'HTML태그'
                    WHEN regexp_matches(r.text, '&(amp|quot|lt|gt|nbsp|#[0-9]+);') THEN '특수문자코드'
                    WHEN regexp_matches(r.text, '(https?://|www\\.)') THEN 'URL'
               END AS 종류
        FROM raw r
        JOIN txt t ON t.review_id = substr(md5(r.user_id || '|' || r.asin || '|' || CAST(r.timestamp AS VARCHAR)), 1, 16)
        WHERE length(r.text) < 400
    )
    SELECT 종류, 원문, 정리본 FROM j WHERE 종류 IS NOT NULL
    QUALIFY row_number() OVER (PARTITION BY 종류 ORDER BY hash(원문)) <= 3
    ORDER BY 종류
""").df()
examples

## 6. [점검] 품목별 요약

- `빈_본문`: 감정 분석에서 빠질 리뷰
- `길이조건_충족`: 10~100단어 리뷰 수와 비율. 논문 표본 조건을 적용하면 이만큼 남는다
- `단어수_중앙값`, `상위90%`: 리뷰 길이 분포
- `비ASCII_많음`: `high_non_ascii` 리뷰 수

In [ ]:
summary = con.sql("""
    SELECT product_type,
           count(*) AS 리뷰,
           sum(is_empty::INT) AS 빈_본문,
           sum(in_len_range::INT) AS 길이조건_충족,
           round(100.0 * sum(in_len_range::INT) / count(*), 1) AS 충족_비율,
           median(n_words) AS 단어수_중앙값,
           quantile_cont(n_words, 0.9) AS 단어수_상위90,
           sum(high_non_ascii::INT) AS 비ASCII_많음
    FROM txt GROUP BY 1 ORDER BY 1
""").df()
summary

## 7. [점검] 비ASCII 글자가 많은 리뷰 예시

`high_non_ascii`로 표시된 리뷰가 어떤 글인지(이모지만 있는 글, 다른 문자 언어 등) 확인한다. 결과를 보고 표본에서 뺄지 정한다.

In [ ]:
con.sql("""
    SELECT product_type, round(non_ascii_share, 2) AS 비ASCII_비율, left(text_clean, 150) AS 본문
    FROM txt WHERE high_non_ascii
    ORDER BY hash(review_id) LIMIT 10
""").df()

## 8. 남은 중복 확인 (표시만)

논문은 중복 제거를 한다. 9/30에 `user_id + asin + timestamp + text`가 같은 리뷰는 이미 지웠다. 여기서는 **같은 사람이 같은 본문을 다른 ASIN(색상·사이즈 옵션 등)에 올린 경우**를 센다. Amazon은 옵션 상품끼리 리뷰를 공유해서 이런 경우가 생길 수 있다. 감정 분석 표본에서 같은 글이 두 번 뽑히지 않도록 `dup_same_user_text` 열로 표시한다 (첫 번째만 False).

In [ ]:
con.sql("""
    ALTER TABLE txt ADD COLUMN dup_same_user_text BOOLEAN;
    UPDATE txt SET dup_same_user_text = d.rn > 1
    FROM (SELECT review_id, row_number() OVER (PARTITION BY user_id, text_clean ORDER BY review_date, review_id) AS rn
          FROM txt WHERE NOT is_empty) d
    WHERE txt.review_id = d.review_id;
    UPDATE txt SET dup_same_user_text = false WHERE dup_same_user_text IS NULL;
""")
con.sql("""
    SELECT product_type, sum(dup_same_user_text::INT) AS 같은사람_같은본문_중복,
           round(100.0 * sum(dup_same_user_text::INT) / count(*), 2) AS 비율
    FROM txt GROUP BY 1 ORDER BY 1
""").df()

## 9. 저장

이후 노트북(표본 추출, EDA)은 이 파일을 쓴다. 용량이 커서 git에는 올리지 않는다 (`data/` 폴더는 .gitignore 대상).

In [ ]:
con.sql(f"COPY (SELECT * FROM txt ORDER BY product_type, review_date) TO '{OUT_PATH.as_posix()}' (FORMAT parquet)")
print("저장:", OUT_PATH)
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{OUT_PATH.as_posix()}')").df()[["column_name", "column_type"]]

## 10. 로그에 붙일 문장

아래 출력을 복사해 `logs/preprocessing_log.md` 맨 아래에 붙인다.

In [ ]:
tot = con.sql("""SELECT count(*), sum(is_empty::INT), sum(in_len_range::INT), sum(high_non_ascii::INT),
                       sum(dup_same_user_text::INT) FROM txt""").fetchone()
lines = [
    "### 2026-10-06 T1. 리뷰 텍스트 전처리 (김지우)",
    "노트북: `notebooks/03_text_preprocess.ipynb`. 기준: Attri et al. (2025) Findings of EMNLP 전처리·표본 조건",
    "",
    "| 항목 | 결정 | 이유 |",
    "|---|---|---|",
    "| 본문 정리 | `<br>` 등 → 공백, 나머지 HTML 태그·URL 삭제, 특수문자 코드 복원, 공백 정리 | 논문의 review text standardizing. 형식만 정리 |",
    "| 하지 않은 것 | 소문자·문장부호·불용어·표제어 변환 안 함 | LLM 입력은 원문 그대로가 좋고, trigger가 본문의 정확한 부분 문자열이어야 함 (p.5517) |",
    "| 길이 | 단어 수 10~100이면 `in_len_range` 표시 (삭제 안 함) | 논문 표본 조건 (p.5519). 표본 추출 단계에서 적용 |",
    "| 모델 입력 | 본문(`text_clean`)만. 평점·도움돼요 수 제외 | 논문이 편향 때문에 제외 (p.5519) |",
    "| review_id | md5(user_id\\|asin\\|timestamp) 앞 16자리 | 결과를 원래 리뷰에 다시 붙이기 위한 고정 ID |",
    "| 비ASCII | 이모지·악센트 등 비ASCII 글자 비율 ≥ 0.3이면 `high_non_ascii` 표시 (삭제 안 함) | LLM이 읽기 어려운 글 표시. 언어 판별은 표본에만 적용 |",
    "| 중복 | 같은 사람·같은 본문이 다른 ASIN에 있으면 두 번째부터 `dup_same_user_text` 표시 | 옵션 상품끼리 리뷰 공유. 표본에서 같은 글 중복 방지 |",
    "",
    f"- 전체 {tot[0]:,}건 / 빈 본문 {tot[1]:,}건 / 10~100단어 {tot[2]:,}건 ({100*tot[2]/tot[0]:.1f}%) / 비ASCII 많음 {tot[3]:,}건 / 같은 사람·같은 본문 중복 {tot[4]:,}건",
]
print("\n".join(lines))